In [ ]:
li = data["li"]
li_all = data["li_all"]

st.subheader("Product Performance")

# Style Tags are cross-cutting attributes (No-Show, Loafer, Gift, Kids, etc.)
# that any Product Family can carry. Filtering here narrows every chart below
# to only line items carrying at least one of the selected tags.
all_tags = sorted({t.strip() for tags in li["Style Tags"].dropna() for t in tags.split(",") if t.strip()})
selected_tags = st.multiselect("Filter by Style Tag", all_tags, key="product_style_tags")
if selected_tags:
    tag_mask = li["Style Tags"].apply(lambda tags: any(t in tags for t in selected_tags))
    li = li[tag_mask]
    tag_mask_all = li_all["Style Tags"].apply(lambda tags: any(t in tags for t in selected_tags))
    li_all = li_all[tag_mask_all]

# State / Pincode filters narrow every chart below to a specific shipping
# geography, the same way the Style Tag filter does.
geo_col1, geo_col2 = st.columns(2)
with geo_col1:
    all_states = sorted(li["Shipping Province"].dropna().unique()) if "Shipping Province" in li.columns else []
    selected_states = st.multiselect("Filter by State", all_states, key="product_states")
with geo_col2:
    pincode_pool = li[li["Shipping Province"].isin(selected_states)] if selected_states else li
    all_pincodes = sorted(pincode_pool["Shipping Zip"].dropna().unique()) if "Shipping Zip" in li.columns else []
    selected_pincodes = st.multiselect("Filter by Pincode", all_pincodes, key="product_pincodes")

if selected_states:
    li = li[li["Shipping Province"].isin(selected_states)]
    li_all = li_all[li_all["Shipping Province"].isin(selected_states)]
if selected_pincodes:
    li = li[li["Shipping Zip"].isin(selected_pincodes)]
    li_all = li_all[li_all["Shipping Zip"].isin(selected_pincodes)]

if li.empty:
    st.info("No products match the selected filters.")
else:
    n_families = li["Product Family"].nunique()
    total_units = li["Lineitem quantity"].sum()
    total_revenue = li["Revenue"].sum()
    top_family = li.groupby("Product Family")["Revenue"].sum().idxmax()

    c1, c2, c3, c4 = st.columns(4)
    with c1:
        kpi_card("Product Families", f"{n_families:,}")
    with c2:
        kpi_card("Units Sold (Paid)", f"{total_units:,.0f}")
    with c3:
        kpi_card("Line-item Revenue", f"₹{total_revenue:,.0f}")
    with c4:
        kpi_card("Top Product Family", top_family)

In [ ]:
if not li.empty:
    metric_label = st.radio("View Top Families by", ["Revenue", "Units Sold", "Orders"],
                              horizontal=True, key="product_top_metric")
    if metric_label == "Revenue":
        agg = li.groupby("Product Family")["Revenue"].sum()
    elif metric_label == "Units Sold":
        agg = li.groupby("Product Family")["Lineitem quantity"].sum()
    else:
        agg = li.groupby("Product Family")["Name"].nunique()

    top_fam = agg.sort_values(ascending=False).head(10).reset_index()
    top_fam.columns = ["Product Family", metric_label]

    col1, col2 = st.columns(2)
    with col1:
        fig = px.bar(top_fam, x=metric_label, y="Product Family", orientation="h",
                      title=f"Top 10 Product Families by {metric_label}", color_discrete_sequence=[ACCENT1])
        fig.update_layout(yaxis=dict(autorange="reversed"))
        st.plotly_chart(fig, width='stretch')

    with col2:
        if "Shipping City" in li.columns:
            top_city_rev = li.dropna(subset=["Shipping City"]).groupby("Shipping City")["Revenue"].sum().sort_values(ascending=False).head(10).reset_index()
            fig = px.bar(top_city_rev, x="Revenue", y="Shipping City", orientation="h",
                          title="Top 10 Cities by Revenue", color_discrete_sequence=[ACCENT2])
            fig.update_layout(yaxis=dict(autorange="reversed"))
            st.plotly_chart(fig, width='stretch')

In [ ]:
if not li.empty and "Order Month" in li.columns:
    st.subheader("Product Family Trend Over Time")
    fam_revenue = li.groupby("Product Family")["Revenue"].sum().sort_values(ascending=False)

    tcol1, tcol2 = st.columns([3, 1])
    with tcol1:
        sel_families = st.multiselect("Product Families", fam_revenue.index.tolist(),
                                        default=fam_revenue.head(6).index.tolist(), key="product_trend_families")
    with tcol2:
        trend_metric = st.radio("Metric", ["Revenue", "Units Sold"], key="product_trend_metric")

    if sel_families:
        metric_col = "Revenue" if trend_metric == "Revenue" else "Lineitem quantity"
        trend = li[li["Product Family"].isin(sel_families)].groupby(["Order Month", "Product Family"])[metric_col].sum().reset_index()
        trend["Order Month"] = trend["Order Month"].astype(str)
        fig = px.line(trend, x="Order Month", y=metric_col, color="Product Family", markers=True,
                       title=f"Monthly {trend_metric} Trend")
        st.plotly_chart(fig, width='stretch')
    else:
        st.info("Select at least one product family to see its trend.")

In [ ]:
if not li.empty and "Pack Size" in li.columns and li["Pack Size"].notna().any():
    st.subheader("Pack Size Analysis")

    pack_basis = st.radio(
        "Pack Size Basis",
        ["SKU Pack Size", "Effective Bundle Size"],
        horizontal=True, key="product_pack_basis",
        help=(
            "SKU Pack Size is the pack size printed on the line item itself "
            "(e.g. 'Pack of 1-Pair', 'Pack of 5-Pair'). Many 'Pack of 1-Pair' SKUs are "
            "actually sold as part of a multi-buy offer -- the customer adds several "
            "1-pair items to the same order. Effective Bundle Size rolls those up to "
            "the total number of 1-pair items bought together in one order, which "
            "better reflects what the customer actually walked away with."
        ),
    )
    size_col = "Pack Size" if pack_basis == "SKU Pack Size" else "Effective Pack Size"

    def _bucket_pack_size(s):
        bucket = s.clip(upper=10).astype(int).astype(str) + "-pair"
        bucket = bucket.where(s < 10, "10+ pairs")
        return bucket

    bucket_order = [f"{n}-pair" for n in range(1, 10)] + ["10+ pairs"]

    pcol1, pcol2 = st.columns(2)

    with pcol1:
        pack_metric = st.radio("Metric", ["Revenue", "Units Sold", "Avg Price per Pair"],
                                 horizontal=True, key="product_pack_metric")
        pack_df = li.dropna(subset=[size_col]).copy()
        pack_df["Pack Bucket"] = _bucket_pack_size(pack_df[size_col])
        if pack_metric == "Revenue":
            agg = pack_df.groupby("Pack Bucket")["Revenue"].sum().reset_index(name="Value")
            y_title = "Revenue (₹)"
        elif pack_metric == "Units Sold":
            agg = pack_df.groupby("Pack Bucket")["Lineitem quantity"].sum().reset_index(name="Value")
            y_title = "Units Sold"
        else:
            pack_df["Price per Pair"] = pack_df["Lineitem price"] / pack_df["Pack Size"]
            agg = pack_df.groupby("Pack Bucket")["Price per Pair"].mean().reset_index(name="Value")
            y_title = "Avg Price per Pair (₹)"
        agg["Pack Bucket"] = pd.Categorical(agg["Pack Bucket"], categories=bucket_order, ordered=True)
        agg = agg.sort_values("Pack Bucket")
        fig = px.bar(agg, x="Pack Bucket", y="Value", title=f"{pack_metric} by {pack_basis}",
                      color_discrete_sequence=[PALETTE[2]])
        fig.update_layout(yaxis_title=y_title, xaxis_title=pack_basis)
        st.plotly_chart(fig, width='stretch')

    with pcol2:
        if (not li_all.empty and {size_col, "Is Shipped", "Any RTO", "Order Month"}.issubset(li_all.columns)):
            shipped_pack = li_all[(li_all["Is Shipped"] == True) & (li_all["Order Month"] >= RTO_TRACKING_START)]
            shipped_pack = shipped_pack.dropna(subset=[size_col]).copy()
            shipped_pack["Pack Bucket"] = _bucket_pack_size(shipped_pack[size_col])
            pack_rto = shipped_pack.groupby("Pack Bucket").agg(
                Shipped_Orders=("Name", "nunique"), RTO_Orders=("Any RTO", "sum")
            )
            pack_rto = pack_rto[pack_rto["Shipped_Orders"] >= 30]
            if not pack_rto.empty:
                pack_rto["RTO Rate %"] = (pack_rto["RTO_Orders"] / pack_rto["Shipped_Orders"] * 100).round(2)
                pack_rto = pack_rto.reset_index()
                pack_rto["Pack Bucket"] = pd.Categorical(pack_rto["Pack Bucket"], categories=bucket_order, ordered=True)
                pack_rto = pack_rto.sort_values("Pack Bucket")
                fig = px.bar(pack_rto, x="Pack Bucket", y="RTO Rate %", title=f"RTO Rate by {pack_basis}",
                              color_discrete_sequence=[DANGER])
                fig.update_layout(xaxis_title=pack_basis)
                st.plotly_chart(fig, width='stretch')
                st.caption(
                    f"Limited to orders shipped from {RTO_TRACKING_START} onward, with at least 30 "
                    "shipped orders per bucket."
                )

    if pack_basis == "Effective Bundle Size":
        ones = li.dropna(subset=["Pack Size"])
        ones = ones[ones["Pack Size"] == 1]
        if not ones.empty:
            common_bundle = ones["Effective Pack Size"].mode().iloc[0]
            st.caption(
                f"**Effective Bundle Size** treats every 'Pack of 1-Pair' line item as part of the "
                f"multi-buy offer it was ordered in -- e.g. an order with six separate 1-pair items "
                f"shows up here as a 6-pair bundle. The most common bundle size for these items is "
                f"**{int(common_bundle)}-pair**."
            )

In [ ]:
# Returns / RTO by category
if not li_all.empty and "Is Shipped" in li_all.columns and "Any RTO" in li_all.columns:
    st.subheader("Delivery Performance by Product Family")
    min_orders = st.slider("Minimum shipped orders threshold", 5, 200, 30, key="product_min_orders")
    shipped_li = li_all[li_all["Is Shipped"] == True]
    if "Order Month" in shipped_li.columns:
        shipped_li = shipped_li[shipped_li["Order Month"] >= RTO_TRACKING_START]
    overall_rto_rate = shipped_li["Any RTO"].mean() * 100
    cat_delivery = shipped_li.groupby("Product Family").agg(
        Shipped_Orders=("Name", "nunique"),
        RTO_Orders=("Any RTO", "sum"),
    )
    cat_delivery = cat_delivery[cat_delivery["Shipped_Orders"] >= min_orders]
    cat_delivery["RTO Rate %"] = (cat_delivery["RTO_Orders"] / cat_delivery["Shipped_Orders"] * 100).round(2)
    cat_delivery["Delivery Rate %"] = (100 - cat_delivery["RTO Rate %"]).round(2)
    cat_delivery = cat_delivery.sort_values("RTO Rate %", ascending=False)
    st.caption(
        f"Limited to orders shipped from {RTO_TRACKING_START} onward. Earlier orders almost never "
        "carry an 'RTO Initiated'/'RTO Delivered' tag (the tagging process wasn't consistently used "
        "yet), so including them would make older/niche categories look like they have a falsely "
        "perfect 100% delivery rate."
    )

    col1, col2 = st.columns(2)
    with col1:
        worst = cat_delivery.head(10).reset_index()
        fig = px.bar(worst, x="RTO Rate %", y="Product Family", orientation="h",
                      title="Highest RTO Rate Product Families", color_discrete_sequence=[DANGER])
        fig.update_layout(yaxis=dict(autorange="reversed"))
        st.plotly_chart(fig, width='stretch')
    with col2:
        best = cat_delivery.tail(10).sort_values("Delivery Rate %", ascending=False).reset_index()
        fig = px.bar(best, x="Delivery Rate %", y="Product Family", orientation="h",
                      title="Best Delivery Rate Product Families", color_discrete_sequence=[SUCCESS])
        fig.update_layout(yaxis=dict(autorange="reversed"))
        st.plotly_chart(fig, width='stretch')

    if not cat_delivery.empty:
        st.markdown("###")
        impact = cat_delivery.reset_index().sort_values("RTO_Orders", ascending=False).head(15)
        fig = px.scatter(impact, x="Shipped_Orders", y="RTO Rate %", size="RTO_Orders",
                          color="RTO Rate %", hover_name="Product Family", color_continuous_scale="Reds",
                          title="RTO Impact: Shipped Volume vs RTO Rate (bubble size = returned orders)")
        fig.add_hline(y=overall_rto_rate, line_dash="dash", line_color=PALETTE[4],
                       annotation_text=f"Overall avg: {overall_rto_rate:.1f}%")
        st.plotly_chart(fig, width='stretch')
        top_impact = impact.iloc[0]
        st.caption(
            f"**{top_impact['Product Family']}** has the largest absolute RTO impact: "
            f"**{int(top_impact['RTO_Orders']):,}** returned orders out of "
            f"{int(top_impact['Shipped_Orders']):,} shipped ({top_impact['RTO Rate %']:.1f}%). "
            "Families toward the top-right (high volume *and* above the dashed overall-average "
            "line) are the biggest opportunities for cutting total returns."
        )

In [ ]:
if not li.empty:
    st.markdown("---")
    st.subheader("Product Family Deep Dive")
    families_sorted = li.groupby("Product Family")["Revenue"].sum().sort_values(ascending=False).index.tolist()
    sel_family = st.selectbox("Select a Product Family", families_sorted, key="product_deep_dive")

    fam_li = li[li["Product Family"] == sel_family]

    dc1, dc2, dc3, dc4 = st.columns(4)
    with dc1:
        kpi_card("Revenue", f"₹{fam_li['Revenue'].sum():,.0f}")
    with dc2:
        kpi_card("Units Sold", f"{fam_li['Lineitem quantity'].sum():,.0f}")
    with dc3:
        kpi_card("Orders", f"{fam_li['Name'].nunique():,}")
    with dc4:
        kpi_card("Avg Item Price", f"₹{fam_li['Lineitem price'].mean():,.0f}")

    dcol1, dcol2 = st.columns(2)
    with dcol1:
        if "Order Month" in fam_li.columns:
            trend = fam_li.groupby("Order Month")["Revenue"].sum().reset_index()
            trend["Order Month"] = trend["Order Month"].astype(str)
            fig = px.line(trend, x="Order Month", y="Revenue", markers=True,
                           title=f"{sel_family}: Monthly Revenue")
            fig.update_traces(line_color=ACCENT1, fill="tozeroy", fillcolor="rgba(91,141,239,0.10)")
            st.plotly_chart(fig, width='stretch')

    with dcol2:
        if "Shipping City" in fam_li.columns:
            top_cities = fam_li.dropna(subset=["Shipping City"]).groupby("Shipping City")["Revenue"].sum().sort_values(ascending=False).head(8).reset_index()
            fig = px.bar(top_cities, x="Revenue", y="Shipping City", orientation="h",
                          title=f"{sel_family}: Top Cities by Revenue", color_discrete_sequence=[ACCENT2])
            fig.update_layout(yaxis=dict(autorange="reversed"))
            st.plotly_chart(fig, width='stretch')

    if not li_all.empty and {"Is Shipped", "Any RTO", "Order Month"}.issubset(li_all.columns):
        fam_li_all = li_all[li_all["Product Family"] == sel_family]
        fam_shipped = fam_li_all[(fam_li_all["Is Shipped"] == True) & (fam_li_all["Order Month"] >= RTO_TRACKING_START)]
        if fam_shipped["Name"].nunique() >= 10:
            fam_rto_rate = fam_shipped["Any RTO"].mean() * 100
            delta = fam_rto_rate - overall_rto_rate
            comparison = "higher" if delta > 0 else "lower"
            st.caption(
                f"**{sel_family}** RTO rate: **{fam_rto_rate:.1f}%** vs the overall average of "
                f"**{overall_rto_rate:.1f}%** ({abs(delta):.1f} pts {comparison}, orders shipped "
                f"from {RTO_TRACKING_START} onward)."
            )
        else:
            st.caption("Not enough shipped orders for this family to compute a reliable RTO rate.")